In [9]:
import pandas as pd
from datetime import datetime, timedelta
import math
import os

In [10]:
def fetch_openmeteo_archive(lat=23.5948,lon=120.442,start="2014-02-12",end="2014-04-08"):
    OPEN_METEO_API = "https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start}&end_date={end}&hourly=temperature_2m,relativehumidity_2m,precipitation,windspeed_10m,winddirection_10m&timezone=Asia%2FSingapore"
    url = OPEN_METEO_API.format(lat=lat,lon=lon,start=start,end=end)
    df = pd.read_json(url)
    df_obs = pd.DataFrame()
    for index, row in df.iterrows():
        df_obs[index] = row['hourly']
    df_obs['time'] = pd.to_datetime(df_obs['time'])
    #Calculate u, v wind components
    df_obs['u'] = df_obs['windspeed_10m'] * df_obs['winddirection_10m'].apply(lambda x: math.cos(math.radians(270-x)))
    df_obs['v'] = df_obs['windspeed_10m'] * df_obs['winddirection_10m'].apply(lambda x: math.sin(math.radians(270-x)))
    #移除有NaN的資料
    df_obs = df_obs.dropna()
    return df_obs

def fetch_openmeteo_forecast(lat=23.5948, lon=120.442, past_days=7, forecast_days=16):
    OPEN_METEO_API = "https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,relativehumidity_2m,precipitation,windspeed_10m,winddirection_10m&past_days={past_days}&forecast_days={forecast_days}&models=ecmwf_aifs025&timezone=Asia%2FSingapore"
    url = OPEN_METEO_API.format(lat=lat,lon=lon,past_days=past_days,forecast_days=forecast_days)
    df = pd.read_json(url)
    df_forecast = pd.DataFrame()
    for index, row in df.iterrows():
        df_forecast[index] = row['hourly']
    df_forecast['time'] = pd.to_datetime(df_forecast['time'])
    #Calculate u, v wind components
    df_forecast['u'] = df_forecast['windspeed_10m'] * df_forecast['winddirection_10m'].apply(lambda x: math.cos(math.radians(270-x)))
    df_forecast['v'] = df_forecast['windspeed_10m'] * df_forecast['winddirection_10m'].apply(lambda x: math.sin(math.radians(270-x)))
    #移除有NaN的資料
    df_forecast = df_forecast.dropna()
    return df_forecast

In [11]:
#取得要下載的氣象站列表
STA_LIST = "https://raw.githubusercontent.com/Raingel/weather_station_list/refs/heads/main/data/weather_sta_list.csv"
df_sta = pd.read_csv(STA_LIST)
#僅保留撤站日期為nan的資料
df_sta = df_sta[df_sta['撤站日期'].isna()]
#只保留站號、站名、緯度、經度
df_sta = df_sta[['站號','站名','緯度','經度']]
#移除重複的資料
df_sta = df_sta.drop_duplicates()
print(f"共有 {len(df_sta)} 個有效氣象站")

共有 751 個有效氣象站


In [12]:
lat = 23.5948
lon = 120.442
past_days_start = (datetime.now() - timedelta(days=50)).strftime("%Y-%m-%d")
past_days_end = datetime.now().strftime("%Y-%m-%d")
for index, row in df_sta.iterrows():
    lat = row['緯度']
    lon = row['經度']
    weather_df_archive = fetch_openmeteo_archive(lat=lat, lon=lon, start=past_days_start, end=past_days_end)
    weather_df_forecast = fetch_openmeteo_forecast(lat=lat, lon=lon)
    weather_df = pd.concat([weather_df_archive, weather_df_forecast])
    weather_df = weather_df.drop_duplicates(subset=['time'])
    # Save the data to a CSV file
    #42HA10_萬大發電廠_23.978875_121.139639.csv
    filename = f"{row['站號']}_{row['站名']}_{lat}_{lon}.csv"
    #weather_df.to_csv(filename, index=False)
    print(f"已下載 {row['站號']}_{row['站名']}_{lat}_{lon}.csv")
    #save
    weather_df.to_csv(f"../ERA5/{filename}", index=False)

已下載 466850_五分山雷達站_25.071182_121.781205.csv
已下載 466881_新北_24.9593_121.52.csv
已下載 466900_淡水_25.164889_121.448906.csv
已下載 466910_鞍部_25.182586_121.529731.csv
已下載 466920_臺北_25.037658_121.514853.csv
已下載 466930_竹子湖_25.162078_121.544547.csv
已下載 466940_基隆_25.133314_121.740475.csv
已下載 466950_彭佳嶼_25.627975_122.079744.csv
已下載 466990_花蓮_23.975128_121.613275.csv
已下載 467050_新屋_25.006744_121.047486.csv
已下載 467080_宜蘭_24.763975_121.756528.csv
已下載 467110_金門_24.407306_118.289281.csv
已下載 467270_田中_23.873803_120.581286.csv
已下載 467280_後龍_24.648563_120.831834.csv
已下載 467290_古坑_23.633639_120.551889.csv
已下載 467300_東吉島_23.25695_119.667467.csv
已下載 467350_澎湖_23.565503_119.563094.csv
已下載 467410_臺南_22.993239_120.204772.csv
已下載 467420_永康_23.038386_120.2367.csv
已下載 467441_高雄_22.7304_120.3125.csv
已下載 467480_嘉義_23.495925_120.432906.csv
已下載 467490_臺中_24.145736_120.684075.csv
已下載 467530_阿里山_23.508208_120.813242.csv
已下載 467540_大武_22.355675_120.903789.csv
已下載 467550_玉山_23.487614_120.959522.csv
已下載 467571_新竹_24.827853_121.01